In [1]:
!pip install tldextract Levenshtein python-whois

#하는 일: pip install tldextract Levenshtein python-whois로 필요한 파이썬 연장(라이브러리)을
#컴퓨터에 다운로드함.
#이유: URL을 쪼개고, 글자 수를 세고, 도메인 정보를 조회하는 기능이 기본 파이썬에는 없기 때문.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.5/107.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.5 MB/s eta 0:00:00


In [2]:
import tldextract
ext = tldextract.extract("http://naver-login.com/verify")
print(ext.subdomain, ext.domain, ext.suffix)

#하는 일: [http://naver-login.com/verify](http://naver-login.com/verify)에서 naver-login이라는 메인 도메인만 잘라냄.
#이유: 주소 전체에서 진짜 도메인 이름만 골라내야 브랜드 도용 여부를 판단할 수 있음.

 naver-login com


In [3]:
import Levenshtein
print(Levenshtein.distance("naver-login", "naver"))
print(Levenshtein.distance("taiegram", "telegram"))

#하는 일: naver-login과 naver 사이의 글자 차이 수치(6)를 계산함.
#이유: 진짜 브랜드 이름과 유사한 철자로 속이는 피싱 공격을 잡아내기 위한 기초 수치 확인.

6
2


In [4]:
import whois
try:
    w = whois.whois("google.com")
    print(w.creation_date)
except Exception as e:
    print("조회 실패:", e)

    #하는 일: 인터넷 등록 장부에서 해당 도메인이 언제 만들어졌는지 날짜를 가져옴.
    #이유: 최근에 급하게 만들어진 도메인일수록 피싱 위험도가 높기 때문.

[datetime.datetime(1997, 9, 15, 4, 0, tzinfo=tzoffset('UTC', 0)), datetime.datetime(1997, 9, 15, 7, 0, tzinfo=tzoffset('UTC', 0))]


In [8]:
#하는 일: 셀 2~4에서 검증한 기술들을 합치고 점수를 계산해서,
#README에 팀원들과 약속했던 type, input, score, reasons, unavailable 형태의 JSON 데이터로 출력함.
#이유: 보안A가 분석한 이 결과를 B(화면 담당)가 받아 화면에 노출하고, C(DB 담당)가 서버 장부에
#기록하기 때문에 표준 바통 규격으로 만드는 것!

import tldextract
import json

def analyze_url_sample1(url):
    reasons = []

    # 1. 도메인 잘라내기
    ext = tldextract.extract(url)
    domain_name = ext.domain # naver-login

    # 2. [점수 요인 1] 도메인 유사 (35점)
    # naver라는 진짜 이름이 들어있는데 완벽히 같지 않은 경우
    target_brand = "naver"
    if target_brand in domain_name and domain_name != target_brand:
        reasons.append({
            "factor": "도메인 유사",
            "points": 35,
            "text": f"유명 포털('{target_brand}') 이름을 변형/교묘하게 섞어 쓴 도메인입니다."
        })

    # 3. [점수 요인 2] 긴급 키워드 (10점)
    if "verify" in url or "login" in url:
        reasons.append({
            "factor": "긴급 키워드",
            "points": 10,
            "text": "인증 또는 로그인을 강요하는 긴급성 키워드가 포함되어 있습니다."
        })

    # 4. 점수 합산 (35 + 10 = 45점)
    total_score = sum(r["points"] for r in reasons)

    # 5. 팀 약속 양식(JSON) 생성
    return {
        "type": "url",
        "input": url,
        "score": total_score,
        "model_probability": 0.91,
        "reasons": reasons,
        "unavailable": []
    }

# 실행 및 결과 출력
result = analyze_url_sample1("http://naver-login.com/verify")
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "type": "url",
  "input": "http://naver-login.com/verify",
  "score": 45,
  "model_probability": 0.91,
  "reasons": [
    {
      "factor": "도메인 유사",
      "points": 35,
      "text": "유명 포털('naver') 이름을 변형/교묘하게 섞어 쓴 도메인입니다."
    },
    {
      "factor": "긴급 키워드",
      "points": 10,
      "text": "인증 또는 로그인을 강요하는 긴급성 키워드가 포함되어 있습니다."
    }
  ],
  "unavailable": []
}


점수 계산 구조:
도메인 유사: naver-login에 naver가 포함되어 35점 추가

긴급 키워드: URL에 verify와 login이 들어있어 10점 추가

최종 score: 35 + 10 = 45점 (README 약속 규칙 1: "score는 reasons의 points를 더한 값" 완벽 충족)

[셀 1]: !pip install tldextract Levenshtein python-whois (연장 가져오기)   
[셀 2]: tldextract 도메인 분리 테스트 (연장 점검 1)   [셀 3]: Levenshtein 글자 수 차이 테스트 (연장 점검 2)   
[셀 4]: whois 등록일 조회 테스트 (연장 점검 3)   
[셀 5]: 지금 보여준 코드 (분석 엔진 합체 + 약속된 JSON 바통 출력)